# N1 — Identificación de metáforas primarias (Visualización)
## AI-MELT: Nivel 1 — Exploración visual

**Prerequisito:** Haber ejecutado `N1_metafora_primaria.ipynb` (procesamiento).

---

### Archivos de entrada esperados (en `./outputs/N1/`)

| Archivo | Columnas clave | Descripción |
|---|---|---|
| `n1_metaforas_{enfoque}.csv` | ID_expresion, ID_oracion, expresion_metaforica, foco, foco_part_of_speech, significado_contextual, significado_basico, dominio_fuente, dominio_meta, metafora_conceptual, enfoque, confianza_modelo | Un CSV por enfoque ejecutado |
| `n1_metaforas_primarias.csv` | (mismas columnas + confianza_cross_enfoques) | Consolidado de todos los enfoques |
| `n1_correspondencias_ontologicas.csv` | ID_correspondencia, ID_expresion, enfoque, elemento_fuente, elemento_meta, evidencia_textual | Mapeos ontológicos |
| `n1_correspondencias_epistemicas.csv` | ID_correspondencia_ep, ID_expresion, enfoque, tipo_inferencia, relacion_fuente, inferencia_meta | Inferencias tipificadas |
| `n1_comparacion_enfoques.csv` | enfoque_A, enfoque_B, kappa_oracion, dominios_compartidos | Métricas pairwise |

### Archivo de N0 también usado

| Archivo | Uso |
|---|---|
| `n0_corpus.csv` | Para unir con capítulo/volumen en visualizaciones |

---

### Configuración clave
- `ENFOQUES_A_VISUALIZAR` — qué enfoques incluir en las gráficas (subconjunto de los disponibles en disco)
- `COLORES_ENFOQUES` — mapeo de colores consistente por enfoque

### Visualizaciones generadas
1. Metáforas por capítulo y enfoque
2. Top 20 dominios fuente/meta agregados
3. Top 20 dominios fuente separados por enfoque
4. Top 20 dominios meta separados por enfoque
5. Heatmap dominio fuente × meta
6. POS del foco por enfoque
7. Wordcloud de metáforas conceptuales
8. Correspondencias epistémicas por tipo y enfoque
9. Sankey por cada enfoque
10. Sankey consolidado multi-enfoque
11. Matriz de concordancia entre enfoques

## 1. Configuración y carga

In [ ]:
# ============================================================
# 1. IMPORTS Y CONFIGURACIÓN
# ============================================================
import os
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
import seaborn as sns
from wordcloud import WordCloud

warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11
sns.set_style("whitegrid")

# ┌─────────────────────────────────────────────────────────┐
# │  CONFIGURA QUÉ ENFOQUES VISUALIZAR                      │
# └─────────────────────────────────────────────────────────┘

ENFOQUES_A_VISUALIZAR = ["claude", "openai", "huggingface", "embeddings"]

COLORES_ENFOQUES = {
    "claude": "#3B8BD4",
    "openai": "#D85A30",
    "huggingface": "#1D9E75",
    "embeddings": "#7F77DD",
}

N0_DIR = "./outputs/N0/"
N1_DIR = "./outputs/N1/"
VIZ_DIR = "./outputs/N1/viz/"
os.makedirs(VIZ_DIR, exist_ok=True)

# Cargar N0 para metadatos de capítulo/volumen
df_n0 = pd.read_csv(os.path.join(N0_DIR, "n0_corpus.csv"),
                      usecols=["ID_oracion", "capitulo", "volumen"] if "volumen" in
                      pd.read_csv(os.path.join(N0_DIR, "n0_corpus.csv"), nrows=0).columns
                      else ["ID_oracion", "capitulo"])
print(f"✓ N0 metadatos: {len(df_n0):,} oraciones")

# Cargar resultados por enfoque
resultados_disponibles = {}
for enfoque in ENFOQUES_A_VISUALIZAR:
    path = os.path.join(N1_DIR, f"n1_metaforas_{enfoque}.csv")
    if os.path.exists(path):
        resultados_disponibles[enfoque] = pd.read_csv(path)
        print(f"✓ n1_metaforas_{enfoque}.csv: {len(resultados_disponibles[enfoque]):,} filas")
    else:
        print(f"⚠ n1_metaforas_{enfoque}.csv no encontrado — omitido")

ENFOQUES_DISPONIBLES = list(resultados_disponibles.keys())
print(f"\nEnfoques que SE visualizarán: {ENFOQUES_DISPONIBLES}")

if not ENFOQUES_DISPONIBLES:
    print("\n⚠ No hay datos. Ejecuta primero N1_metafora_primaria.ipynb")

## 12. Visualizaciones exploratorias

Todas las visualizaciones iteran sobre `ENFOQUES_DISPONIBLES` (los que tienen CSV en disco). Si un enfoque no está, se omite limpiamente sin romper el notebook.

### 12.1 Metáforas por capítulo y enfoque

In [ ]:
# ============================================================
# 12.1 METÁFORAS POR CAPÍTULO (un subplot por enfoque)
# ============================================================

if ENFOQUES_DISPONIBLES:
    n_enf = len(ENFOQUES_DISPONIBLES)
    n_cols = min(n_enf, 2)
    n_rows = (n_enf + 1) // 2
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(8*n_cols, 5*n_rows))
    if n_enf == 1:
        axes = [axes]
    elif n_rows > 1 or n_cols > 1:
        axes = np.array(axes).flatten()

    for i, enfoque in enumerate(ENFOQUES_DISPONIBLES):
        df_sub = resultados_disponibles[enfoque]
        ax = axes[i] if n_enf > 1 else axes[0]
        if len(df_sub) > 0 and "capitulo" in df_n0.columns:
            df_joined = df_sub.merge(df_n0[["ID_oracion", "capitulo"]].drop_duplicates(),
                                      on="ID_oracion", how="left")
            cap_counts = df_joined["capitulo"].value_counts().sort_index()
            ax.bar(cap_counts.index.astype(str), cap_counts.values,
                    color=COLORES_ENFOQUES.get(enfoque, "#888"))
        ax.set_xlabel("Capítulo")
        ax.set_ylabel("Número de metáforas")
        ax.set_title(f"Metáforas por capítulo — {enfoque}")

    # Ocultar subplots vacíos
    for j in range(n_enf, len(axes) if isinstance(axes, np.ndarray) else 1):
        axes[j].set_visible(False)

    plt.tight_layout()
    plt.savefig(os.path.join(VIZ_DIR, "viz_metaforas_por_capitulo.png"), dpi=150, bbox_inches='tight')
    plt.show()
else:
    print("⚠ Sin enfoques disponibles para visualizar.")

### 12.2 Top 20 dominios fuente y meta AGREGADOS (todos los enfoques)

In [ ]:
# ============================================================
# 12.2 TOP 20 DOMINIOS AGREGADOS (original, sin separar enfoques)
# ============================================================

if ENFOQUES_DISPONIBLES:
    df_all = pd.concat([resultados_disponibles[e] for e in ENFOQUES_DISPONIBLES], ignore_index=True)
    df_all_nonempty = df_all[df_all["dominio_fuente"].notna() & (df_all["dominio_fuente"] != "")]

    if len(df_all_nonempty) > 0:
        fig, axes = plt.subplots(1, 2, figsize=(16, 8))
        top_fuente = df_all_nonempty["dominio_fuente"].value_counts().head(20)
        axes[0].barh(top_fuente.index[::-1], top_fuente.values[::-1], color="#1D9E75")
        axes[0].set_xlabel("Frecuencia")
        axes[0].set_title(f"Top 20 dominios fuente (AGREGADO de {len(ENFOQUES_DISPONIBLES)} enfoques)")

        top_meta = df_all_nonempty["dominio_meta"].value_counts().head(20)
        axes[1].barh(top_meta.index[::-1], top_meta.values[::-1], color="#7F77DD")
        axes[1].set_xlabel("Frecuencia")
        axes[1].set_title(f"Top 20 dominios meta (AGREGADO de {len(ENFOQUES_DISPONIBLES)} enfoques)")

        plt.tight_layout()
        plt.savefig(os.path.join(VIZ_DIR, "viz_top_dominios.png"), dpi=150, bbox_inches='tight')
        plt.show()
    else:
        print("⚠ Ningún enfoque disponible produce dominios (ninguno extrae componentes completos).")

### 12.3 Top 20 dominios fuente SEPARADOS por enfoque

Grid dinámico: una columna por cada enfoque en `ENFOQUES_DISPONIBLES`.

In [ ]:
# ============================================================
# 12.3 TOP 20 DOMINIOS FUENTE POR ENFOQUE
# ============================================================

# Solo los enfoques que producen dominios (no solo foco)
enfoques_con_dominios = [e for e in ENFOQUES_DISPONIBLES
                          if resultados_disponibles[e]["dominio_fuente"].notna().any()
                          and (resultados_disponibles[e]["dominio_fuente"] != "").any()]

if enfoques_con_dominios:
    n_cols = min(len(enfoques_con_dominios), 4)
    n_rows = (len(enfoques_con_dominios) + n_cols - 1) // n_cols
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(6*n_cols, 6*n_rows))
    if len(enfoques_con_dominios) == 1:
        axes = [axes]
    else:
        axes = np.array(axes).flatten()

    for i, enfoque in enumerate(enfoques_con_dominios):
        df_sub = resultados_disponibles[enfoque]
        df_sub = df_sub[df_sub["dominio_fuente"].notna() & (df_sub["dominio_fuente"] != "")]
        top = df_sub["dominio_fuente"].value_counts().head(20)
        axes[i].barh(top.index[::-1], top.values[::-1], color=COLORES_ENFOQUES.get(enfoque, "#888"))
        axes[i].set_xlabel("Frecuencia")
        axes[i].set_title(f"Top 20 dominios FUENTE — {enfoque}")
        axes[i].tick_params(axis='y', labelsize=8)

    for j in range(len(enfoques_con_dominios), len(axes)):
        if len(axes) > 1:
            axes[j].set_visible(False)

    plt.tight_layout()
    plt.savefig(os.path.join(VIZ_DIR, "viz_top_dominios_fuente_por_enfoque.png"),
                 dpi=150, bbox_inches='tight')
    plt.show()
else:
    print("⚠ Ningún enfoque disponible produce dominios fuente.")

### 12.4 Top 20 dominios meta SEPARADOS por enfoque

In [ ]:
# ============================================================
# 12.4 TOP 20 DOMINIOS META POR ENFOQUE
# ============================================================

if enfoques_con_dominios:
    n_cols = min(len(enfoques_con_dominios), 4)
    n_rows = (len(enfoques_con_dominios) + n_cols - 1) // n_cols
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(6*n_cols, 6*n_rows))
    if len(enfoques_con_dominios) == 1:
        axes = [axes]
    else:
        axes = np.array(axes).flatten()

    for i, enfoque in enumerate(enfoques_con_dominios):
        df_sub = resultados_disponibles[enfoque]
        df_sub = df_sub[df_sub["dominio_meta"].notna() & (df_sub["dominio_meta"] != "")]
        top = df_sub["dominio_meta"].value_counts().head(20)
        axes[i].barh(top.index[::-1], top.values[::-1], color=COLORES_ENFOQUES.get(enfoque, "#888"))
        axes[i].set_xlabel("Frecuencia")
        axes[i].set_title(f"Top 20 dominios META — {enfoque}")
        axes[i].tick_params(axis='y', labelsize=8)

    for j in range(len(enfoques_con_dominios), len(axes)):
        if len(axes) > 1:
            axes[j].set_visible(False)

    plt.tight_layout()
    plt.savefig(os.path.join(VIZ_DIR, "viz_top_dominios_meta_por_enfoque.png"),
                 dpi=150, bbox_inches='tight')
    plt.show()

### 12.5 Heatmap dominio fuente × dominio meta (agregado)

In [ ]:
# ============================================================
# 12.5 HEATMAP DE MAPEOS (AGREGADO)
# ============================================================

if ENFOQUES_DISPONIBLES:
    df_all = pd.concat([resultados_disponibles[e] for e in ENFOQUES_DISPONIBLES], ignore_index=True)
    df_heat_src = df_all[df_all["dominio_fuente"].notna() & (df_all["dominio_fuente"] != "")]
    if len(df_heat_src) > 0:
        top_f = df_heat_src["dominio_fuente"].value_counts().head(15).index
        top_m = df_heat_src["dominio_meta"].value_counts().head(15).index
        df_heat = df_heat_src[df_heat_src["dominio_fuente"].isin(top_f) &
                               df_heat_src["dominio_meta"].isin(top_m)]
        cross = pd.crosstab(df_heat["dominio_fuente"], df_heat["dominio_meta"])
        fig, ax = plt.subplots(figsize=(14, 10))
        sns.heatmap(cross, annot=True, fmt="d", cmap="YlOrRd", ax=ax, linewidths=0.5)
        ax.set_title("Heatmap: Dominio fuente × Dominio meta (Top 15, agregado)")
        ax.set_xlabel("Dominio meta")
        ax.set_ylabel("Dominio fuente")
        plt.tight_layout()
        plt.savefig(os.path.join(VIZ_DIR, "viz_heatmap_dominios.png"), dpi=150, bbox_inches='tight')
        plt.show()

### 12.6 POS del foco por enfoque

In [ ]:
# ============================================================
# 12.6 POS DEL FOCO POR ENFOQUE
# ============================================================

enfoques_con_pos = [e for e in ENFOQUES_DISPONIBLES
                     if resultados_disponibles[e]["foco_part_of_speech"].notna().any()
                     and (resultados_disponibles[e]["foco_part_of_speech"] != "").any()]

if enfoques_con_pos:
    n_cols = min(len(enfoques_con_pos), 4)
    n_rows = (len(enfoques_con_pos) + n_cols - 1) // n_cols
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(5*n_cols, 5*n_rows))
    if len(enfoques_con_pos) == 1:
        axes = [axes]
    else:
        axes = np.array(axes).flatten()

    pos_colors = {"VERB": "#3B8BD4", "NOUN": "#1D9E75", "ADJ": "#D85A30",
                   "ADV": "#7F77DD", "PROPN": "#E8A838"}

    for i, enfoque in enumerate(enfoques_con_pos):
        df_sub = resultados_disponibles[enfoque]
        df_sub = df_sub[df_sub["foco_part_of_speech"].notna() &
                         (df_sub["foco_part_of_speech"] != "")]
        pos_counts = df_sub["foco_part_of_speech"].value_counts()
        axes[i].pie(pos_counts.values, labels=pos_counts.index, autopct='%1.1f%%',
                     colors=[pos_colors.get(p, "#888") for p in pos_counts.index],
                     startangle=90)
        axes[i].set_title(f"POS del foco — {enfoque}")

    for j in range(len(enfoques_con_pos), len(axes)):
        if len(axes) > 1:
            axes[j].set_visible(False)

    plt.tight_layout()
    plt.savefig(os.path.join(VIZ_DIR, "viz_pos_foco.png"), dpi=150, bbox_inches='tight')
    plt.show()

### 12.7 Wordcloud de metáforas conceptuales

In [ ]:
# ============================================================
# 12.7 WORDCLOUD DE METÁFORAS CONCEPTUALES
# ============================================================

if ENFOQUES_DISPONIBLES:
    df_all = pd.concat([resultados_disponibles[e] for e in ENFOQUES_DISPONIBLES], ignore_index=True)
    mc_series = df_all["metafora_conceptual"].dropna()
    mc_series = mc_series[mc_series != ""]

    if len(mc_series) > 0:
        mc_text = " ".join(mc_series.tolist())
        wc = WordCloud(width=1200, height=500, background_color="white",
                        max_words=80, colormap="magma", collocations=False).generate(mc_text)
        fig, ax = plt.subplots(figsize=(14, 6))
        ax.imshow(wc, interpolation="bilinear")
        ax.axis("off")
        ax.set_title("Metáforas conceptuales identificadas (agregado)", fontsize=14)
        plt.tight_layout()
        plt.savefig(os.path.join(VIZ_DIR, "viz_wordcloud_metaforas.png"),
                     dpi=150, bbox_inches='tight')
        plt.show()
    else:
        print("⚠ Sin metáforas conceptuales para generar wordcloud.")

### 12.8 Correspondencias epistémicas por tipo

Solo para enfoques que las producen (LLMs directos o híbridos).

In [ ]:
# ============================================================
# 12.8 CORRESPONDENCIAS EPISTÉMICAS POR TIPO
# ============================================================

path_epi = os.path.join(OUTPUT_DIR, "n1_correspondencias_epistemicas.csv")
if os.path.exists(path_epi):
    df_epi = pd.read_csv(path_epi)
    df_epi = df_epi[df_epi["enfoque"].isin(ENFOQUES_DISPONIBLES)]

    if len(df_epi) > 0:
        fig, ax = plt.subplots(figsize=(12, 6))
        tipo_enfoque_counts = df_epi.groupby(["tipo_inferencia", "enfoque"]).size().unstack(fill_value=0)
        tipo_enfoque_counts.plot(kind="bar", ax=ax,
                                   color=[COLORES_ENFOQUES.get(e, "#888")
                                          for e in tipo_enfoque_counts.columns])
        ax.set_xlabel("Tipo de inferencia")
        ax.set_ylabel("Frecuencia")
        ax.set_title("Correspondencias epistémicas por tipo y enfoque")
        plt.xticks(rotation=45, ha='right')
        plt.legend(title="Enfoque")
        plt.tight_layout()
        plt.savefig(os.path.join(VIZ_DIR, "viz_correspondencias_epistemicas.png"),
                     dpi=150, bbox_inches='tight')
        plt.show()
else:
    print("⚠ No hay correspondencias epistémicas disponibles.")

### 12.9 Sankey por cada enfoque

Estructura: **dominio_meta (izquierda) → metáfora_conceptual (centro) → dominio_fuente (derecha)**

Un HTML interactivo por enfoque.

In [ ]:
# ============================================================
# 12.9 SANKEY POR ENFOQUE
# ============================================================

def build_sankey_per_enfoque(df_sub, enfoque_label, top_n=20):
    """Sankey: meta → metáfora_conceptual → fuente."""
    df_clean = df_sub[
        df_sub["dominio_fuente"].notna() & (df_sub["dominio_fuente"] != "") &
        df_sub["dominio_meta"].notna() & (df_sub["dominio_meta"] != "") &
        df_sub["metafora_conceptual"].notna() & (df_sub["metafora_conceptual"] != "")
    ]
    if len(df_clean) == 0:
        return None

    # Tomar top N metáforas conceptuales más frecuentes
    top_mc = df_clean["metafora_conceptual"].value_counts().head(top_n).index.tolist()
    df_filt = df_clean[df_clean["metafora_conceptual"].isin(top_mc)]

    # Construir nodos únicos: metas + metáforas_conceptuales + fuentes
    metas = df_filt["dominio_meta"].unique().tolist()
    mcs = df_filt["metafora_conceptual"].unique().tolist()
    fuentes = df_filt["dominio_fuente"].unique().tolist()

    all_labels = [f"[META] {m}" for m in metas] + mcs + [f"[FTE] {f}" for f in fuentes]
    node_index = {lbl: i for i, lbl in enumerate(all_labels)}

    # Links: meta → mc
    sources = []
    targets = []
    values = []
    for mc in mcs:
        df_mc = df_filt[df_filt["metafora_conceptual"] == mc]
        for meta in df_mc["dominio_meta"].unique():
            count = len(df_mc[df_mc["dominio_meta"] == meta])
            sources.append(node_index[f"[META] {meta}"])
            targets.append(node_index[mc])
            values.append(count)
        for fte in df_mc["dominio_fuente"].unique():
            count = len(df_mc[df_mc["dominio_fuente"] == fte])
            sources.append(node_index[mc])
            targets.append(node_index[f"[FTE] {fte}"])
            values.append(count)

    color_enf = COLORES_ENFOQUES.get(enfoque_label, "#888888")

    fig = go.Figure(data=[go.Sankey(
        node=dict(pad=10, thickness=15, line=dict(color="black", width=0.5),
                  label=all_labels),
        link=dict(source=sources, target=targets, value=values,
                  color=color_enf + "88")
    )])
    fig.update_layout(title=f"Sankey — enfoque {enfoque_label} "
                             f"(meta → metáfora conceptual → fuente, top {top_n})",
                       font_size=10, height=700)
    return fig


enfoques_sankey = [e for e in ENFOQUES_DISPONIBLES
                    if resultados_disponibles[e]["dominio_fuente"].notna().any()
                    and (resultados_disponibles[e]["dominio_fuente"] != "").any()]

for enfoque in enfoques_sankey:
    fig = build_sankey_per_enfoque(resultados_disponibles[enfoque], enfoque, top_n=20)
    if fig:
        out_path = os.path.join(OUTPUT_DIR, f"viz_sankey_{enfoque}.html")
        pio.write_html(fig, out_path, include_plotlyjs="cdn")
        print(f"✓ {out_path}")
        fig.show()
    else:
        print(f"⚠ Sin datos de dominios para Sankey de {enfoque}")

if not enfoques_sankey:
    print("⚠ Ningún enfoque disponible produce dominios completos para Sankey.")

### 12.10 Sankey consolidado multi-enfoque

Estructura: **dominio_meta (izquierda) → dominio_fuente (derecha)**  
El ancho del flujo = frecuencia agregada; el color del flujo refleja el enfoque dominante en ese par (meta, fuente).

In [ ]:
# ============================================================
# 12.10 SANKEY CONSOLIDADO (meta → fuente, coloreado por enfoque dominante)
# ============================================================

def build_sankey_consolidado(resultados_dict, enfoques_list, top_n=25):
    dfs_combined = []
    for e in enfoques_list:
        df = resultados_dict[e].copy()
        if df["dominio_fuente"].notna().any() and (df["dominio_fuente"] != "").any():
            dfs_combined.append(df)
    if not dfs_combined:
        return None

    df_all = pd.concat(dfs_combined, ignore_index=True)
    df_all = df_all[
        df_all["dominio_fuente"].notna() & (df_all["dominio_fuente"] != "") &
        df_all["dominio_meta"].notna() & (df_all["dominio_meta"] != "")
    ]
    if len(df_all) == 0:
        return None

    # Agregar por (meta, fuente): contar instancias y determinar enfoque dominante
    group = df_all.groupby(["dominio_meta", "dominio_fuente", "enfoque"]).size().reset_index(name="count")
    pair_totals = group.groupby(["dominio_meta", "dominio_fuente"])["count"].sum().reset_index()
    pair_totals = pair_totals.sort_values("count", ascending=False).head(top_n)

    top_pairs = set(zip(pair_totals["dominio_meta"], pair_totals["dominio_fuente"], strict=False))
    group_filtered = group[group.apply(
        lambda r: (r["dominio_meta"], r["dominio_fuente"]) in top_pairs, axis=1)]

    # Determinar enfoque dominante para cada par
    dominant_by_pair = {}
    for (meta, fte), sub in group_filtered.groupby(["dominio_meta", "dominio_fuente"]):
        dominant_by_pair[(meta, fte)] = sub.loc[sub["count"].idxmax(), "enfoque"]

    # Construir nodos y links
    metas = sorted(set(m for m, f in top_pairs))
    fuentes = sorted(set(f for m, f in top_pairs))
    all_labels = [f"[META] {m}" for m in metas] + [f"[FTE] {f}" for f in fuentes]
    node_index = {lbl: i for i, lbl in enumerate(all_labels)}

    sources, targets, values, colors = [], [], [], []
    for (meta, fte) in top_pairs:
        total = pair_totals[(pair_totals["dominio_meta"] == meta) &
                             (pair_totals["dominio_fuente"] == fte)]["count"].iloc[0]
        sources.append(node_index[f"[META] {meta}"])
        targets.append(node_index[f"[FTE] {fte}"])
        values.append(int(total))
        dom_enf = dominant_by_pair[(meta, fte)]
        colors.append(COLORES_ENFOQUES.get(dom_enf, "#888888") + "AA")

    fig = go.Figure(data=[go.Sankey(
        node=dict(pad=10, thickness=18, line=dict(color="black", width=0.5),
                   label=all_labels),
        link=dict(source=sources, target=targets, value=values, color=colors)
    )])
    fig.update_layout(
        title=f"Sankey CONSOLIDADO — dominio META → dominio FUENTE "
              f"(top {top_n} pares, color = enfoque dominante)",
        font_size=10, height=800
    )
    return fig


fig_cons = build_sankey_consolidado(resultados_disponibles, ENFOQUES_DISPONIBLES, top_n=25)
if fig_cons:
    out_path = os.path.join(VIZ_DIR, "viz_sankey_consolidado.html")
    pio.write_html(fig_cons, out_path, include_plotlyjs="cdn")
    print(f"✓ {out_path}")

    # Leyenda de colores
    print("\nLeyenda de colores por enfoque dominante:")
    for e in ENFOQUES_DISPONIBLES:
        print(f"  ■ {e}: {COLORES_ENFOQUES.get(e, '#888')}")
    fig_cons.show()
else:
    print("⚠ Sin datos suficientes para Sankey consolidado.")

### 12.11 Matriz de concordancia entre enfoques

In [ ]:
# ============================================================
# 12.11 HEATMAP DE CONCORDANCIA (Cohen's kappa)
# ============================================================

if kappa_matrix is not None and len(ENFOQUES_DISPONIBLES) >= 2:
    fig, ax = plt.subplots(figsize=(8, 6))
    sns.heatmap(kappa_matrix.astype(float), annot=True, fmt=".3f",
                 cmap="RdYlGn", center=0.5, vmin=-0.2, vmax=1.0, ax=ax,
                 cbar_kws={"label": "Cohen's κ"})
    ax.set_title("Concordancia entre enfoques (Cohen's κ a nivel de oración)")
    plt.tight_layout()
    plt.savefig(os.path.join(VIZ_DIR, "viz_matriz_concordancia.png"),
                 dpi=150, bbox_inches='tight')
    plt.show()

    print("\nInterpretación de Cohen's κ:")
    print("  < 0.00: sin acuerdo")
    print("  0.00-0.20: leve")
    print("  0.21-0.40: aceptable")
    print("  0.41-0.60: moderado")
    print("  0.61-0.80: sustancial")
    print("  0.81-1.00: casi perfecto")
else:
    print("⚠ No hay matriz de concordancia (se requieren ≥2 enfoques).")

## 13. Resumen

In [ ]:
# ============================================================
# 13. RESUMEN
# ============================================================
print("=" * 60)
print("N1 — VISUALIZACIONES COMPLETADAS")
print("=" * 60)
print(f"\nEnfoques visualizados: {ENFOQUES_DISPONIBLES}")
print(f"\nArchivos en {VIZ_DIR}:")
for f_name in sorted(os.listdir(VIZ_DIR)):
    size = os.path.getsize(os.path.join(VIZ_DIR, f_name)) / 1024
    icon = "📊" if f_name.endswith(".html") else "🖼"
    print(f"  {icon} {f_name} ({size:.0f} KB)")
print("\n➡ SIGUIENTE: Ejecutar N2_convencionalizacion.ipynb")